# phase2 / notebook 4 — extended GAT training

1000-epoch training of GraphAttention at the hyperparameters that were memory-feasible.

# 1. Overview

## 1.1 Task & motivation

Confirm GAT's convergence properties at a long training horizon and identify the point of diminishing returns.

## 1.2 Dataset summary

Reddit2 via `NNGraphDataset` (an nnx wrapper).

## 1.3 Approach in one paragraph

Train GAT with 5 attention heads for 1000 epochs at hidden `[512, 256, 128, 64]`. Inspect validation loss over time; rank this notebook's GAT run; visualize it.

## 1.4 Libraries used

`torch_geometric`, `nnx` (including `NNGraphDataset`, `VisUtils`).

# 2. Environment & Setup

## 2.1 Imports

The setup cell imports `torch_geometric as pyg` and nnx primitives (the rest assumed from sibling notebooks).

## 2.2 Configuration / hyperparameters

`n_headss = [5]`, `n_epochs = 1000`.

## 2.3 Reproducibility (seed, device)

`nnx` defaults.

# 3. Data

## 3.1 Loading

The dataset is loaded with `NNGraphDataset(n_workers=4, ...)`.

## 3.2 Inspection / EDA

Covered in phase 1.

## 3.3 Preprocessing & splits

PyG defaults.

# 4. Model

## 4.1 Architecture

GAT, 5 attention heads, hidden `[512, 256, 128, 64]`.

## 4.2 Loss & optimizer

Cross-entropy. Best optimizer config from earlier notebooks.

## 4.3 Why this design

Single-architecture deep-dive: holds everything else constant and varies training length.

# 5. Training

## 5.1 Training loop

`nnx` loop, 1000 epochs.

## 5.2 Metrics tracked

Per-iteration train/val loss; 37 s/iter average.

## 5.3 Run-time notes

Tier B. Total training time ~10 h 25 m on original M1 Max.

# 6. Evaluation & Results

## 6.1 Test-set evaluation

This notebook's GAT run reaches a final validation error around 0.2232 — a marginal gain over the 500-epoch run. The ranking below is scoped to the `runs` created by this notebook, not the shared cross-experiment registry.

## 6.2 Visualizations

`VisUtils.multi_line_plot` renders this notebook's run loss and learning-rate curves; a later cell (`N_SAMPLES = 10_000`) prepares a held-out sample and t-SNE-projects checkpoint logits for downstream inspection.

## 6.3 Discussion

Diminishing returns past ~500 epochs. Persistent validation-metric fluctuations suggest the optimizer might benefit from a learning-rate schedule.

In [ ]:
# Papermill parameters cell. Default values used when run interactively.
# Set via: papermill -p SMOKE_TEST 1 in.ipynb out.ipynb
# SMOKE_TEST: 1 = run a tiny smoke version of this notebook
SMOKE_TEST = 0
# SMOKE_TEST_EPOCHS: max epochs when SMOKE_TEST=1
SMOKE_TEST_EPOCHS = 1
# SMOKE_TEST_SUBSET: max samples when SMOKE_TEST=1
SMOKE_TEST_SUBSET = 256

In [1]:
import torch_geometric as pyg

from nnx.nn.dataset.nn_graph_dataset import NNGraphDataset

from nnx.nn.enum.nets import Nets
from nnx.nn.enum.losses import Losses
from nnx.nn.enum.devices import Devices

from nnx.nn.params.nn_params import NNParams
from nnx.nn.params.nn_train_params import NNTrainParams
from nnx.nn.params.nn_model_params import NNModelParams

from nnx.nn.nn_model import NNModel

from nnx.utils import Utils
from nnx.vis_utils import VisUtils

In [2]:
def _smoke_subset_masks(subset):
    """Keep only the first `subset` seed nodes per split mask — bounds smoke
    memory + runtime so smoke CI samples small subgraphs instead of the
    full-Reddit single batch that OOMs the runner (issue #25)."""
    def _transform(data):
        for name in ("train_mask", "val_mask", "test_mask"):
            mask = getattr(data, name, None)
            if mask is None:
                continue
            idx = mask.nonzero(as_tuple=True)[0]
            if idx.numel() > subset:
                keep = mask.clone()
                keep[idx[subset:]] = False
                setattr(data, name, keep)
        return data
    return _transform


_transforms = [pyg.transforms.NormalizeFeatures()]
if SMOKE_TEST:
    _transforms.append(_smoke_subset_masks(SMOKE_TEST_SUBSET))

ds = NNGraphDataset(
    n_workers=4,
    n_neighbors=([5, 5] if SMOKE_TEST else [20, 15, 10]),  # smoke: tiny fanout bounds the sampled subgraph (this nb's GAT is large)
    ds_class=pyg.datasets.Reddit2,
    transform=pyg.transforms.Compose(_transforms),
)

Utils.print_table(
    header=False
    , data=ds.state()
    , title='Dataset Details...'
)

+---------------------------------+
|        Dataset Details...       |
+------------------+--------------+
|       name       |   Reddit2    |
|    input_dim     |     602      |
|    output_dim    |      41      |
| train_batch_size |   153,932    |
|  val_batch_size  |    23,699    |
| test_batch_size  |    55,334    |
|    n_workers     |      4       |
|   n_neighbors    | [20, 15, 10] |
+------------------+--------------+


In [ ]:
n_headss = [5]
n_epochs = SMOKE_TEST_EPOCHS if SMOKE_TEST else 1000

dropout_probs = [0.5]
hidden_dimss = [
    [512, 256, 128, 64]
]

models = [
    model
        for model in [
            NNModel(
                params=NNModelParams(
                    net=Nets.GRAPH_ATT
                    , device=Devices.CPU
                    , loss=Losses.CROSS_ENTROPY
                )
                , net_params=NNParams(
                    n_heads=n_heads
                    , hidden_dims=hidden_dims
                    , input_dim=ds.input_dim
                    , output_dim=ds.output_dim
                    , dropout_prob=dropout_prob
                )
            )
                for dropout_prob in dropout_probs
                for hidden_dims in hidden_dimss
                for n_heads in n_headss
        ]
]

train_params = [
    train_param
        for train_param in [
            NNTrainParams(n_epochs=n_epochs)
                .with_train_loader(value=ds.train_loader)
                .with_val_loader(value=ds.val_loader)
        ]
]

runs = [
    run for run in [
        model.train(params=train_param)
            for model in models
            for train_param in train_params
    ]
]

In [3]:
top_runs = [
    run for run in sorted(
        runs
        , key=lambda run: min(
            (idp.val_edp.error for idp in run.idps if idp.val_edp is not None)
            , default=float("inf")
        )
    )[:10]
]

In [ ]:
VisUtils.multi_line_plot(
    x_ticks_inc=50
    , y_axis_label="Error"
    , x_axis_label="Iterations"
    , title="Training & Validation Losses"
    , fig_size=(24, 13.5)
    , x=[
        iter_idx for iter_idx in range(
            0
            , max(
                top_runs
                , key=lambda run: run.idps[-1].iter_idx
            ).idps[-1].iter_idx
        )
    ]
    , yss_legend=(
        [str(run) for run in top_runs]
        , ['Training', 'Validation']
    )
    , yss=[
        [
            [idp.train_edp.error for idp in run.idps if idp.val_edp is not None]
            , [idp.val_edp.error for idp in run.idps if idp.val_edp is not None]
        ] for run in top_runs
    ]
)

In [ ]:
VisUtils.multi_line_plot(
    x_ticks_inc=50
    , y_axis_label="LR"
    , x_axis_label="Iterations"
    , title="Learning Rates"
    , fig_size=(24, 13.5)
    , x=[
        iter_idx for iter_idx in range(
            0
            , max(
                top_runs
                , key=lambda run: run.idps[-1].iter_idx
            ).idps[-1].iter_idx
        )
    ]
    , yss_legend=(
        [str(run) for run in top_runs]
        , ['LR']
    )
    , yss=[
        [
            [idp.lr for idp in run.idps]
        ] for run in top_runs
    ]
)

In [ ]:
print(f"best run is {top_runs[0]} which achieves validation error of {min(top_runs[0].idps, key=lambda idp: idp.val_edp.error).val_edp.error:.4f}")

In [8]:
N_SAMPLES = 10_000

# top_runs[0] is the best run (lowest val error) — the same run runs/best
# points at; checkpoints() yields FIRST/Q1/Q2/Q3/LAST (BEST excluded by design).
for checkpoint in top_runs[0].checkpoints():
    if checkpoint is None:
        continue
    VisUtils.two_dim_tsne_checkpoint_logits(checkpoint=checkpoint, ds=ds, n_samples=N_SAMPLES)